# Step 2-1. DenseNet121 — Abdominal Ultrasound (ABD-US)

## 1. Overview

This notebook demonstrates the training of a DenseNet121 model using abdominal ultrasound (ABD-US) images for type 2 diabetes (T2D) classification and the extraction of image activation vectors for subsequent graph neural network (GNN) and multimodal prediction analyses.

The model is trained from scratch using the demonstration dataset, rather than applying a previously trained model.

## 2. Image Dimensions and Preprocessing

**Image dimensions are reported as Height × Width throughout this notebook.**

| Image stage | Height × Width | Channels |
|---|---|---|
| Stored ABD-US demonstration images | 496 × 685 pixels | Grayscale |
| DenseNet121 model input | 224 × 224 pixels | 1 |

The demonstration images have already undergone image preparation. During model training and inference, images are resized to **224 × 224 pixels** and intensity-scaled to the range [0, 1].

Resizing and model-specific transformations are performed in memory and do not alter the stored demonstration images.

Due to data privacy restrictions, individual-level medical images cannot be publicly shared. Therefore, the original image preprocessing procedures, including image quality control, grayscale conversion, and region-of-interest (ROI) cropping, cannot be demonstrated in this notebook. These procedures are described separately in the documentation.

## 3. Dataset Partition

The demonstration dataset contains 100 participants, divided into four mutually exclusive subsets:

| Dataset | Participants | Purpose |
|---|---:|---|
| Training1 | 32 | DenseNet121 training |
| Training2 | 32 | Image feature extraction for downstream models |
| Validation | 16 | DenseNet121 model selection |
| Testing | 20 | Image feature extraction and subsequent model evaluation |

DenseNet121 is trained exclusively using Training1. The checkpoint with the highest image-level validation AUC is selected using the Validation dataset.

Following training, the selected DenseNet121 model is applied to all four datasets to generate image activation vectors.

## 4. Model Architecture and Hyperparameters

The model uses the DenseNet121 architecture implemented in MONAI, with a single grayscale input channel and two output classes.

The original study implementation used:


| Hyperparameter | Original implementation | GitHub demo |
|---|---|---|
| Architecture | MONAI DenseNet121 | MONAI DenseNet121 |
| Optimizer | Adam | Adam |
| Learning rate | 1 × 10⁻⁵ | **1 × 10⁻⁵** |
| Loss function | Cross-entropy | Cross-entropy |
| Batch size | 64 | **64** (adjustable) |
| Pretrained weights | Not used | **Not used** |
| Maximum training epochs | 500 | **20**, to shorten computation time |
| Validation | Every epoch | Every epoch |
| Model selection | Highest validation AUC | Highest validation AUC |

Training augmentation uses random rotation (±15°), flipping, and zooming (0.9–1.1). Validation and activation-vector extraction use no random augmentation. All controllable parameters appear at the beginning of the notebook.


**For this GitHub demonstration, the maximum training epochs are reduced from 500 to 20 to shorten computational time.**

All hyperparameters are explicitly defined at the beginning of the notebook and can be adjusted according to the user's computational resources.

Training augmentation includes random rotation, flipping, and zooming. Validation and feature extraction do not use random augmentation.

## 5. GPU and Computational Requirements

GPU acceleration is recommended but not restricted to a specific NVIDIA GPU model.

Users can configure GPU selection, batch size, number of data-loading workers, and automatic mixed precision according to their available hardware.

CPU execution is supported as a fallback, although training may be slower.

## 6. Outputs

The notebook produces:

1. The best DenseNet121 model checkpoint selected using validation AUC.
2. Training history, including training loss and validation performance.
3. The configuration used for the demonstration run.
4. A 1024-dimensional activation vector for each ABD-US image.

The activation vectors preserve participant identifiers, dataset partitions, and image filenames for subsequent GNN analysis.

All generated files are saved to `Outputs/Step2_DenseNet/ABD/`. The original files in `Demo_data/` are not modified.

**Note:** Validation AUC in this notebook is calculated at the image level for checkpoint selection and does not represent the participant-level performance of the final multimodal T2D prediction model.


## Environment Setup

This notebook requires PyTorch and MONAI for DenseNet121 training and image feature extraction. A dedicated GPU environment is provided in `environments/gpu.yml` to support reproducibility.

### 1. Create the GPU environment

Open a Terminal and navigate to the root directory of the downloaded GitHub repository.

Run the following command:

```bash
conda env create -f environments/gpu.yml
```

This command installs the required software packages, including PyTorch, MONAI, and other dependencies.

**Note:** Environment installation is required only once. If `t2d-gpu` has already been created, this step can be skipped.

### 2. Register the Jupyter Kernel

Run the following command in the Terminal:

```bash
conda run -n t2d-gpu python -m ipykernel install \
    --user \
    --name t2d-gpu \
    --display-name "T2D GPU"
```

### 3. Verify the installation

Run the following command in the Terminal:

```bash
conda run -n t2d-gpu python -c \
    "import torch, monai; print(torch.__version__, monai.__version__)"
```

### 4. Select the Jupyter Kernel

Open this notebook in JupyterLab or Jupyter Notebook.

Select:

**Kernel → Change Kernel → T2D GPU**

The notebook can then be executed sequentially, starting from **Cell 1 — User-defined Hyperparameters**.

### GPU Compatibility

GPU acceleration is recommended for DenseNet121 training. The environment is not restricted to a specific NVIDIA GPU model, but a compatible NVIDIA driver and CUDA-supported device are required for GPU execution.

Users may adjust GPU selection, batch size, and data-loading parameters in Cell 1 according to their computational resources.

The same GPU environment can be reused for all four DenseNet121 image modalities:

- ABD-US
- BMD left hip
- BMD right hip
- BMD spine

No additional package installation is required within the model-training cells.


In [1]:
# ============================================================
# 1. User-defined Hyperparameters — ABD DenseNet121
# ============================================================

from pathlib import Path
import json
import time
import platform

import numpy as np
import pandas as pd
from PIL import Image

import torch
import monai

from monai.networks.nets import DenseNet121
from monai.utils import set_determinism


# ============================================================
# 1.1 Project and Dataset Configuration
# ============================================================

PROJECT_ROOT = Path.cwd().resolve()

MODALITY = "ABD"

IMAGE_DIR = (
    PROJECT_ROOT
    / "Demo_data/1-2.Sample_image/ABD"
)

SAMPLE_INFO_FILE = (
    PROJECT_ROOT
    / "Demo_data/1-1.Sample_information/sample_information.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "Outputs/Step2_DenseNet/ABD"
)


# ============================================================
# 1.2 Image Configuration
# ============================================================

# Image dimensions: Height x Width
# Stored demonstration images: 496 x 685 pixels
SOURCE_IMAGE_HW = (496, 685)

# DenseNet121 input: 224 x 224 pixels
INPUT_HW = (224, 224)

# Grayscale image
IN_CHANNELS = 1

# Binary classification: Control / T2D
NUM_CLASSES = 2


# ============================================================
# 1.3 Model Hyperparameters
# ============================================================

MODEL_NAME = "DenseNet121"

# Original implementation: training from scratch
PRETRAINED = False

# Original study: 500 epochs
# GitHub demonstration: 20 epochs to reduce computation time
MAX_EPOCHS = 20

# Learning rate from the original DenseNet implementation
LEARNING_RATE = 1e-5

# Optimizer and loss
OPTIMIZER = "Adam"
LOSS_FUNCTION = "CrossEntropyLoss"

# Batch size
TRAIN_BATCH_SIZE = 64
EVAL_BATCH_SIZE = 64

# Validation every epoch
VAL_INTERVAL = 1

# Reproducibility
SEED = 0


# ============================================================
# 1.4 Data Augmentation
# ============================================================

# Rotation: +/- 15 degrees
ROTATION_DEGREES = 15
ROTATION_PROB = 0.5

# Random flip
FLIP_AXIS = 0
FLIP_PROB = 0.5

# Random zoom
ZOOM_RANGE = (0.9, 1.1)
ZOOM_PROB = 0.5


# ============================================================
# 1.5 GPU / Computational Configuration
# ============================================================

# Index of the GPU visible to the current process
GPU_INDEX = 0

# Allow CPU execution when CUDA is unavailable
ALLOW_CPU_FALLBACK = True

# Data loading
NUM_WORKERS = 2

# Optional mixed precision
# False follows the original implementation
USE_AMP = False


# ============================================================
# 1.6 Output Configuration
# ============================================================

CHECKPOINT_FILE = (
    OUTPUT_DIR / "ABD_best_model.pth"
)

HISTORY_FILE = (
    OUTPUT_DIR / "ABD_TrainingHistory.csv"
)

CONFIG_FILE = (
    OUTPUT_DIR / "ABD_RunConfig.json"
)

FEATURE_FILE = (
    OUTPUT_DIR / "ABD_ActivationVectors.csv"
)


# ============================================================
# 1.7 Device Selection
# ============================================================

if torch.cuda.is_available():

    if GPU_INDEX >= torch.cuda.device_count():
        raise ValueError(
            f"GPU_INDEX={GPU_INDEX}, but only "
            f"{torch.cuda.device_count()} GPUs are visible."
        )

    DEVICE = torch.device(f"cuda:{GPU_INDEX}")

else:

    if not ALLOW_CPU_FALLBACK:
        raise RuntimeError(
            "CUDA is unavailable and CPU fallback is disabled."
        )

    DEVICE = torch.device("cpu")


# Set random seed
set_determinism(seed=SEED)

# Verify input data
assert SAMPLE_INFO_FILE.is_file(), SAMPLE_INFO_FILE
assert IMAGE_DIR.is_dir(), IMAGE_DIR

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1.8 Display Configuration
# ============================================================

print("=" * 65)
print("ABD DenseNet121 — DEMONSTRATION CONFIGURATION")
print("=" * 65)

print("Model:", MODEL_NAME)
print("Modality:", MODALITY)

print("\nImage dimensions (Height x Width)")
print("Stored images:", SOURCE_IMAGE_HW)
print("Model input:", INPUT_HW)
print("Input channels:", IN_CHANNELS)

print("\nTraining parameters")
print("Optimizer:", OPTIMIZER)
print("Learning rate:", LEARNING_RATE)
print("Epochs:", MAX_EPOCHS)
print("Batch size:", TRAIN_BATCH_SIZE)
print("Loss:", LOSS_FUNCTION)
print("Pretrained:", PRETRAINED)
print("Validation interval:", VAL_INTERVAL)
print("Seed:", SEED)

print("\nSoftware environment")
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("MONAI:", monai.__version__)

print("\nComputational device")
print("Device:", DEVICE)

if DEVICE.type == "cuda":

    gpu_name = torch.cuda.get_device_name(DEVICE)

    gpu_memory = (
        torch.cuda.get_device_properties(DEVICE).total_memory
        / 1024**3
    )

    print("GPU:", gpu_name)
    print("GPU memory (GiB):", round(gpu_memory, 2))

print("\nOutput directory:")
print(OUTPUT_DIR.relative_to(PROJECT_ROOT))

print("\nConfiguration completed.")

ABD DenseNet121 — DEMONSTRATION CONFIGURATION
Model: DenseNet121
Modality: ABD

Image dimensions (Height x Width)
Stored images: (496, 685)
Model input: (224, 224)
Input channels: 1

Training parameters
Optimizer: Adam
Learning rate: 1e-05
Epochs: 20
Batch size: 64
Loss: CrossEntropyLoss
Pretrained: False
Validation interval: 1
Seed: 0

Software environment
Python: 3.11.18
PyTorch: 2.7.1+cu128
MONAI: 1.5.1

Computational device
Device: cuda:0
GPU: NVIDIA A100-SXM4-40GB
GPU memory (GiB): 39.49

Output directory:
Outputs/Step2_DenseNet/ABD

Configuration completed.


In [2]:
# ============================================================
# 2. Load ABD Images and Build DataLoaders
# ============================================================

from torch.utils.data import Dataset, DataLoader

from monai.transforms import (
    Compose,
    Resize,
    ScaleIntensity,
    RandRotate,
    RandFlip,
    RandZoom,
    EnsureType,
)


# ============================================================
# 2.1 Read Sample Information
# ============================================================

sample_info = pd.read_csv(
    SAMPLE_INFO_FILE,
    dtype={"Demo_ID": str}
)

assert len(sample_info) == 100
assert sample_info["Demo_ID"].is_unique

# Expected participant-level partitions
expected_splits = {
    "training1": 32,
    "training2": 32,
    "validation": 16,
    "testing": 20,
}

actual_splits = (
    sample_info["Dataset"]
    .value_counts()
    .to_dict()
)

assert actual_splits == expected_splits, (
    f"Unexpected dataset partitions: {actual_splits}"
)


# ============================================================
# 2.2 Prepare T2D Labels
# ============================================================

label_text = (
    sample_info["T2D"]
    .astype(str)
    .str.strip()
    .str.lower()
)

if set(label_text.unique()) <= {"0", "1"}:

    sample_info["Label"] = label_text.astype(int)

elif set(label_text.unique()) <= {"case", "control"}:

    sample_info["Label"] = (
        label_text
        .map({
            "control": 0,
            "case": 1
        })
        .astype(int)
    )

else:

    raise ValueError(
        f"Unexpected T2D labels: {label_text.unique().tolist()}"
    )


# ============================================================
# 2.3 Build Image-level Manifest
# ============================================================

records = []

for row in sample_info.itertuples(index=False):

    image_files = sorted(
        IMAGE_DIR.glob(
            f"{row.Demo_ID}_ABD_*.jpg"
        )
    )

    expected_n = int(row.N_ABD)

    assert len(image_files) == expected_n, (
        f"Image count mismatch for {row.Demo_ID}: "
        f"expected {expected_n}, found {len(image_files)}"
    )

    for image_file in image_files:

        records.append({
            "Demo_ID": row.Demo_ID,
            "Dataset": row.Dataset,
            "Label": int(row.Label),
            "Image_path": image_file,
            "Image_file": str(
                image_file.relative_to(PROJECT_ROOT)
            ),
        })


image_df = pd.DataFrame(records)

assert not image_df.empty
assert len(image_df) == int(sample_info["N_ABD"].sum())

print("=" * 65)
print("ABD IMAGE DATASET")
print("=" * 65)

print("Participants:", len(sample_info))
print("Total ABD images:", len(image_df))

print("\nDataset distribution:")

display(
    image_df.groupby("Dataset")
    .agg(
        Participants=("Demo_ID", "nunique"),
        Images=("Image_file", "size")
    )
    .reindex(list(expected_splits))
)


# ============================================================
# 2.4 Define Image Transforms
# ============================================================

def build_transforms(training=False):

    transforms = [
        Resize(spatial_size=INPUT_HW),
        ScaleIntensity(),
    ]

    if training:

        transforms.extend([

            RandRotate(
                range_x=np.deg2rad(ROTATION_DEGREES),
                prob=ROTATION_PROB,
                keep_size=True,
            ),

            RandFlip(
                spatial_axis=FLIP_AXIS,
                prob=FLIP_PROB,
            ),

            RandZoom(
                min_zoom=ZOOM_RANGE[0],
                max_zoom=ZOOM_RANGE[1],
                prob=ZOOM_PROB,
                keep_size=True,
            ),

        ])

    transforms.append(EnsureType())

    return Compose(transforms)


train_transform = build_transforms(training=True)
eval_transform = build_transforms(training=False)


# ============================================================
# 2.5 Define ABD Dataset
# ============================================================

class ABDDataset(Dataset):

    def __init__(self, dataframe, transform):

        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, index):

        row = self.df.iloc[index]

        with Image.open(row["Image_path"]) as img:

            # Convert to grayscale
            image = np.asarray(
                img.convert("L"),
                dtype=np.float32
            ).copy()

        # Add channel dimension
        # Height x Width -> Channel x Height x Width
        image = image[None, :, :]

        # Resize, intensity scaling, and augmentation
        image = self.transform(image)

        return (
            image,
            int(row["Label"]),
            row["Demo_ID"],
            row["Image_file"],
        )


# ============================================================
# 2.6 Build DataLoaders
# ============================================================

def get_split(split):

    return image_df.loc[
        image_df["Dataset"] == split
    ].reset_index(drop=True)


def build_loader(dataset, batch_size, shuffle=False):

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
    )


train_ds = ABDDataset(
    get_split("training1"),
    train_transform
)

val_ds = ABDDataset(
    get_split("validation"),
    eval_transform
)

train_loader = build_loader(
    train_ds,
    TRAIN_BATCH_SIZE,
    shuffle=True
)

val_loader = build_loader(
    val_ds,
    EVAL_BATCH_SIZE,
    shuffle=False
)


# ============================================================
# 2.7 Verify Model Input
# ============================================================

image, label, demo_id, image_file = train_ds[0]

assert tuple(image.shape) == (
    IN_CHANNELS,
    INPUT_HW[0],
    INPUT_HW[1],
)

print("\n" + "=" * 65)
print("DATALOADER VERIFICATION")
print("=" * 65)

print("Demo_ID:", demo_id)
print("T2D label:", label)
print("Model input (C x H x W):", tuple(image.shape))

print(
    "Pixel intensity range:",
    round(float(image.min()), 4),
    "to",
    round(float(image.max()), 4)
)

print("\nDataLoaders ready.")

ABD IMAGE DATASET
Participants: 100
Total ABD images: 2487

Dataset distribution:


,Participants,Images
Dataset,,
training1,32,784
training2,32,786
validation,16,385
testing,20,532



DATALOADER VERIFICATION
Demo_ID: D004
T2D label: 1
Model input (C x H x W): (1, 224, 224)
Pixel intensity range: 0.0 to 1.0

DataLoaders ready.


In [3]:
# ============================================================
# 3. Train ABD DenseNet121
# ============================================================

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score
)


# ============================================================
# 3.1 Initialize Model
# ============================================================

model = DenseNet121(
    spatial_dims=2,
    in_channels=IN_CHANNELS,
    out_channels=NUM_CLASSES,
    pretrained=PRETRAINED,
).to(DEVICE)


# ============================================================
# 3.2 Optimizer and Loss Function
# ============================================================

criterion = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

amp_enabled = (
    USE_AMP and DEVICE.type == "cuda"
)

scaler = torch.cuda.amp.GradScaler(
    enabled=amp_enabled
)


# ============================================================
# 3.3 Save Run Configuration
# ============================================================

run_config = {

    "Modality": MODALITY,
    "Model": MODEL_NAME,

    "Stored_image_HW": list(SOURCE_IMAGE_HW),
    "Model_input_HW": list(INPUT_HW),

    "Training_participants": 32,
    "Validation_participants": 16,

    "Max_epochs_demo": MAX_EPOCHS,
    "Max_epochs_original": 500,

    "Optimizer": OPTIMIZER,
    "Learning_rate": LEARNING_RATE,
    "Loss": LOSS_FUNCTION,

    "Train_batch_size": TRAIN_BATCH_SIZE,
    "Eval_batch_size": EVAL_BATCH_SIZE,

    "Pretrained": PRETRAINED,
    "Validation_interval": VAL_INTERVAL,

    "Rotation_degrees": ROTATION_DEGREES,
    "Rotation_prob": ROTATION_PROB,
    "Flip_axis": FLIP_AXIS,
    "Flip_prob": FLIP_PROB,

    "Zoom_range": list(ZOOM_RANGE),
    "Zoom_prob": ZOOM_PROB,

    "Seed": SEED,
    "Num_workers": NUM_WORKERS,
    "Use_AMP": USE_AMP,

    "Device": str(DEVICE),

    "GPU_name": (
        torch.cuda.get_device_name(DEVICE)
        if DEVICE.type == "cuda"
        else None
    ),

    "PyTorch_version": torch.__version__,
    "MONAI_version": monai.__version__,
}

CONFIG_FILE.write_text(
    json.dumps(
        run_config,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


# ============================================================
# 3.4 Training and Validation
# ============================================================

history = []

best_auc = float("-inf")
best_epoch = None

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats(DEVICE)


for epoch in range(1, MAX_EPOCHS + 1):

    epoch_start = time.perf_counter()

    # --------------------------------------------------------
    # Training1
    # --------------------------------------------------------

    model.train()

    train_loss_sum = 0.0
    train_n = 0

    for images, labels, _, _ in train_loader:

        images = images.to(
            DEVICE,
            dtype=torch.float32
        )

        labels = labels.to(
            DEVICE,
            dtype=torch.long
        )

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(
            enabled=amp_enabled
        ):

            logits = model(images)

            loss = criterion(
                logits,
                labels
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_n = len(labels)

        train_loss_sum += (
            loss.item() * batch_n
        )

        train_n += batch_n

    train_loss = (
        train_loss_sum / train_n
    )


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    val_auc = np.nan
    val_accuracy = np.nan

    if epoch % VAL_INTERVAL == 0:

        model.eval()

        val_labels = []
        val_probs = []

        with torch.no_grad():

            for images, labels, _, _ in val_loader:

                images = images.to(
                    DEVICE,
                    dtype=torch.float32
                )

                logits = model(images)

                probs = torch.softmax(
                    logits,
                    dim=1
                )[:, 1]

                val_probs.extend(
                    probs.cpu().numpy().tolist()
                )

                val_labels.extend(
                    labels.numpy().tolist()
                )

        if len(set(val_labels)) != 2:
            raise ValueError(
                "Validation data must contain both T2D classes."
            )

        val_auc = roc_auc_score(
            val_labels,
            val_probs
        )

        val_predictions = (
            np.asarray(val_probs) >= 0.5
        ).astype(int)

        val_accuracy = accuracy_score(
            val_labels,
            val_predictions
        )

        # ----------------------------------------------------
        # Save best checkpoint
        # ----------------------------------------------------

        if val_auc > best_auc:

            best_auc = val_auc
            best_epoch = epoch

            torch.save(
                model.state_dict(),
                CHECKPOINT_FILE
            )


    # --------------------------------------------------------
    # Training History
    # --------------------------------------------------------

    epoch_seconds = (
        time.perf_counter() - epoch_start
    )

    history.append({

        "Epoch": epoch,

        "Training_loss": train_loss,

        "Validation_AUC": val_auc,

        "Validation_accuracy": val_accuracy,

        "Epoch_seconds": epoch_seconds,

    })

    pd.DataFrame(history).to_csv(
        HISTORY_FILE,
        index=False
    )

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Loss={train_loss:.4f} | "
        f"Val AUC={val_auc:.4f} | "
        f"Best AUC={best_auc:.4f} | "
        f"Time={epoch_seconds:.1f}s",
        flush=True
    )


# ============================================================
# 3.5 Training Summary
# ============================================================

print("\n" + "=" * 65)
print("ABD DENSENET TRAINING COMPLETED")
print("=" * 65)

print("Optimizer:", OPTIMIZER)
print("Learning rate:", LEARNING_RATE)
print("Completed epochs:", len(history))

print("\nBest epoch:", best_epoch)
print("Best validation AUC:", round(best_auc, 4))

if DEVICE.type == "cuda":

    peak_memory = (
        torch.cuda.max_memory_allocated(DEVICE)
        / 1024**3
    )

    print(
        "Peak allocated GPU memory (GiB):",
        round(peak_memory, 3)
    )

print(
    "\nBest checkpoint:",
    CHECKPOINT_FILE.relative_to(PROJECT_ROOT)
)

print(
    "Training history:",
    HISTORY_FILE.relative_to(PROJECT_ROOT)
)

print("\nTraining completed successfully.")

[LOCAL_PATH] FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(
[LOCAL_PATH] FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Epoch 01/20 | Loss=0.6875 | Val AUC=0.5739 | Best AUC=0.5739 | Time=8.5s


[LOCAL_PATH] FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Epoch 02/20 | Loss=0.6791 | Val AUC=0.5337 | Best AUC=0.5739 | Time=5.3s


[LOCAL_PATH] FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Epoch 03/20 | Loss=0.6748 | Val AUC=0.4941 | Best AUC=0.5739 | Time=5.0s


... (30 outputs left out)

[LOCAL_PATH] FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Epoch 19/20 | Loss=0.6318 | Val AUC=0.5489 | Best AUC=0.5739 | Time=5.2s


[LOCAL_PATH] FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Epoch 20/20 | Loss=0.6271 | Val AUC=0.5528 | Best AUC=0.5739 | Time=4.8s

ABD DENSENET TRAINING COMPLETED
Optimizer: Adam
Learning rate: 1e-05
Completed epochs: 20

Best epoch: 1
Best validation AUC: 0.5739
Peak allocated GPU memory (GiB): 7.969

Best checkpoint: Outputs/Step2_DenseNet/ABD/ABD_best_model.pth
Training history: Outputs/Step2_DenseNet/ABD/ABD_TrainingHistory.csv

Training completed successfully.


In [4]:
# ============================================================
# 4. Extract ABD Activation Vectors
# ============================================================

# ============================================================
# 4.1 Load Best DenseNet Checkpoint
# ============================================================

if not CHECKPOINT_FILE.is_file():
    raise FileNotFoundError(CHECKPOINT_FILE)

model.load_state_dict(
    torch.load(
        CHECKPOINT_FILE,
        map_location=DEVICE
    )
)

model.eval()


# ============================================================
# 4.2 Extract Features from All Four Partitions
# ============================================================

feature_rows = []

splits = [
    "training1",
    "training2",
    "validation",
    "testing"
]

for split in splits:

    dataset = ABDDataset(
        get_split(split),
        eval_transform
    )

    loader = build_loader(
        dataset,
        EVAL_BATCH_SIZE,
        shuffle=False
    )

    with torch.no_grad():

        for images, labels, demo_ids, filenames in loader:

            images = images.to(
                DEVICE,
                dtype=torch.float32
            )

            # DenseNet feature maps
            feature_maps = model.features(images)

            # Extract activation vectors immediately
            # before the final linear classification layer.
            #
            # ReLU -> Global Average Pooling -> Flatten
            vectors = model.class_layers[:-1](
                feature_maps
            )

            # Final classification layer
            logits = model.class_layers[-1](
                vectors
            )

            probabilities = torch.softmax(
                logits,
                dim=1
            )[:, 1]

            vectors = vectors.cpu().numpy()

            probabilities = (
                probabilities.cpu().numpy()
            )

            for i in range(len(demo_ids)):

                row = {

                    "Demo_ID": demo_ids[i],

                    "Dataset": split,

                    "Image_file": filenames[i],

                    "T2D": int(labels[i]),

                    "Predicted_Probability": float(
                        probabilities[i]
                    ),

                }

                row.update({

                    f"AV_{j:04d}": float(value)

                    for j, value in enumerate(
                        vectors[i]
                    )

                })

                feature_rows.append(row)


# ============================================================
# 4.3 Construct Activation Vector DataFrame
# ============================================================

activation_df = pd.DataFrame(
    feature_rows
)

activation_columns = [
    c for c in activation_df.columns
    if c.startswith("AV_")
]


# ============================================================
# 4.4 Validation
# ============================================================

assert len(activation_df) == len(image_df), (
    "Activation vector count does not match image count."
)

assert len(activation_columns) == 1024, (
    f"Expected 1024 activation dimensions, "
    f"found {len(activation_columns)}"
)

assert (
    activation_df[activation_columns]
    .notna()
    .all()
    .all()
), "Missing activation vector values detected."

assert activation_df["Image_file"].is_unique


# ============================================================
# 4.5 Save Activation Vectors
# ============================================================

activation_df.to_csv(
    FEATURE_FILE,
    index=False
)


# ============================================================
# 4.6 Output Summary
# ============================================================

print("=" * 65)
print("ABD ACTIVATION VECTOR EXTRACTION")
print("=" * 65)

print("Total images:", len(activation_df))

print(
    "Unique participants:",
    activation_df["Demo_ID"].nunique()
)

print(
    "Activation dimensions:",
    len(activation_columns)
)

print("\nImage counts by dataset:")

print(
    activation_df["Dataset"]
    .value_counts()
    .reindex(splits)
    .to_string()
)

print(
    "\nOutput:",
    FEATURE_FILE.relative_to(PROJECT_ROOT)
)

display(
    activation_df[
        [
            "Demo_ID",
            "Dataset",
            "Image_file",
            "Predicted_Probability",
            "AV_0000",
            "AV_0001",
            "AV_0002",
        ]
    ].head()
)

print("\nActivation vector extraction completed.")

ABD ACTIVATION VECTOR EXTRACTION
Total images: 2487
Unique participants: 100
Activation dimensions: 1024

Image counts by dataset:
Dataset
training1     784
training2     786
validation    385
testing       532

Output: Outputs/Step2_DenseNet/ABD/ABD_ActivationVectors.csv


,Demo_ID,Dataset,Image_file,Predicted_Probability,AV_0000,AV_0001,AV_0002
0,D004,training1,Demo_data/1-2.Sample_image/ABD/D004_ABD_001.jpg,0.498927,0.039180,0.619621,0.009296
1,D004,training1,Demo_data/1-2.Sample_image/ABD/D004_ABD_002.jpg,0.503585,0.035233,0.627753,0.009465
2,D004,training1,Demo_data/1-2.Sample_image/ABD/D004_ABD_003.jpg,0.505275,0.072921,0.594015,0.018056
3,D004,training1,Demo_data/1-2.Sample_image/ABD/D004_ABD_004.jpg,0.500111,0.077893,0.596113,0.007191
4,D004,training1,Demo_data/1-2.Sample_image/ABD/D004_ABD_005.jpg,0.505025,0.068170,0.618697,0.000978



Activation vector extraction completed.
